# Here the comes the Selective classifiers


In [17]:
# import the usual suspects
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from src.cf_generator_dice import DiceCFGenerator
from src.data_processor import DataProcessor
from src.model_trainer import ModelTrainer

# import the fancy for bar chart tdqm
from  tqdm.notebook import tqdm,tnrange, tqdm_notebook
import time

import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, roc_auc_score,balanced_accuracy_score

import pickle

In [18]:
import importlib
import src.utils as utl
importlib.reload(utl)

<module 'src.utils' from '/home/vbonsignori/L2loRe/src/utils.py'>

In [19]:
import src.fancy_plots as fplt
importlib.reload(fplt)


<module 'src.fancy_plots' from '/home/vbonsignori/L2loRe/src/fancy_plots.py'>

In [20]:
# let's find the different files we need
config = utl.load_config('config.yaml')
cf_results = config['paths']['counterfactuals']


dt_name = "german_credit"
dt_name = "toy_dataset"

cf_method = 'ils'#dice or 'debug'


name_dataset = {"german_credit": "German Credit", 
                "adult": "Adult",
                "toy_dataset":"Toy Dataset",
                }[dt_name]
cf_results

'counterfactuals/'

In [21]:
data_processor = DataProcessor(config=config)
splits = data_processor.load_splits(dt_name)


Loaded feature names from data/processed/toy_dataset/feature_names.csv
Loaded X_train from data/processed/toy_dataset/X_train.npz
Loaded y_train from data/processed/toy_dataset/y_train.npz
Loaded X_test from data/processed/toy_dataset/X_test.npz
Loaded y_test from data/processed/toy_dataset/y_test.npz
Loaded X_calibration from data/processed/toy_dataset/X_calibration.npz
Loaded y_calibration from data/processed/toy_dataset/y_calibration.npz


In [22]:

model_trainer = ModelTrainer(config)
models = model_trainer.train_model(X_train=splits['X_train'], 
                                    y_train=splits['y_train'], 
                                    X_test=splits['X_test'], 
                                    y_test=splits['y_test'],
                                    dataset_name=dt_name)
print(models.keys())

Processing random_forest for toy_dataset
Loaded random_forest model from models/toy_dataset/random_forest.pkl
Loaded existing random_forest model
Processing mlp for toy_dataset
Loaded mlp model from models/toy_dataset/mlp.pkl
Loaded existing mlp model
Processing xgboost for toy_dataset
Loaded xgboost model from models/toy_dataset/xgboost.pkl
Loaded existing xgboost model
Processing lgbm for toy_dataset
Loaded lgbm model from models/toy_dataset/lgbm.pkl
Loaded existing lgbm model
dict_keys(['random_forest', 'mlp', 'xgboost', 'lgbm'])


## Load the distances files

In [23]:
fname = os.path.join(cf_method,dt_name+"_ranking_latent.pkl")
if utl.check_file_exists(fname):
    with open(fname, "rb") as f:
        all_stats = pickle.load(f)
        print("Loaded",fname)
else:
    print("!"*6,fname, "not found")
fname_test = os.path.join(cf_method,dt_name+"_ranking_test_latent.pkl")
if utl.check_file_exists(fname_test):
    with open(fname_test,'rb') as f:
        all_stats_test = pickle.load(f)
        print("Loaded",fname_test)
else:
    print("!"*6,fname_test, "not found")


Loaded ils/toy_dataset_ranking_latent.pkl
Loaded ils/toy_dataset_ranking_test_latent.pkl


In [24]:
def check_res(stats_dict = all_stats,k="mlp"):
    """this function return True if the distances related to the model k are already computed"""
    # try if the list containing the mean dists has length > 0
    try:
        print("("+str(len(stats_dict[k]['distances']['l2']['mean'])),"elements computed)")
        return len(stats_dict[k]['distances']['l2']['mean']) >= 200
    except Exception as e:
        print(e)
        return False
for k in models.keys():
    print(k, "computed?",check_res(stats_dict = all_stats,k=k))
    
    print("\t","test?",check_res(stats_dict=all_stats_test,k=k))

(200 elements computed)
random_forest computed? True
(250 elements computed)
	 test? True
(200 elements computed)
mlp computed? True
(250 elements computed)
	 test? True
(200 elements computed)
xgboost computed? True
(250 elements computed)
	 test? True
(200 elements computed)
lgbm computed? True
(250 elements computed)
	 test? True


In [26]:

# convert all the list in the dictionary to numpy arrays
for k in models.keys():
    for metric in all_stats[k]['distances'].keys():
        for stat in ["min", "max", "mean", "std"]:
            try:
                all_stats[k]['distances'][metric][stat] = np.array(all_stats[k]['distances'][metric][stat])
            except Exception as e:
                print(e)
                print(k, metric, stat,len(all_stats[k]['distances'][metric][stat]))
                for e in all_stats[k]['distances'][metric][stat]:
                    print(e)
for k in models.keys():
    all_stats[k]['probabs'] = np.array(all_stats[k]['probabs'])
    all_stats[k]['corrects'] = np.array(all_stats[k]['corrects']).reshape(-1)
    all_stats[k]['mean_conf'] = np.array(all_stats[k]['mean_conf'])

In [27]:
# convert all the list in the dictionary to numpy arrays
for k in models.keys():
    for metric in all_stats_test[k]['distances'].keys():
        for stat in ["min", "max", "mean", "std"]:
            try:
                all_stats_test[k]['distances'][metric][stat] = np.array(all_stats_test[k]['distances'][metric][stat])
            except Exception as e:
                print(e)
                print(k, metric, stat,len(all_stats_test[k]['distances'][metric][stat]))
                for e in all_stats_test[k]['distances'][metric][stat]:
                    print(e)
for k in models.keys():
    all_stats_test[k]['probabs'] = np.array(all_stats_test[k]['probabs'])
    all_stats_test[k]['corrects'] = np.array(all_stats_test[k]['corrects']).reshape(-1)
    all_stats_test[k]['mean_conf'] = np.array(all_stats_test[k]['mean_conf'])

#### Here we import the necessary libraries for the 
# Selective classifiers


In [ ]:

#from Lib.L2R.code.model_agnostic import PlugInRule, PlugInRuleAUC, SCRoss

from OLD_src.Lib.L2R.code.model_agnostic import PlugInRule, PlugInRuleAUC, SCRoss
# use importlib as we have to reload the module
#importlib.reload(cfdr)
import rejectmodels.CFDistRejector as cfdr
from src.utils import compute_selective_metrics



In [ ]:
# Selective Classifiers
# the keys are the models
plug_in_rulers = {}
selected_plg = {}

l2_rejectors = {}
l2_rejectors_gamma = {}

plug_in_rulers_auc = {}
selected_plg_auc = {}
selected_l2r = {}
selected_l2r_gamma = {}

# now let's generalize for all the l2r based on distaces:
# since for each metric we have 3 values (min, mean, max) we will have 3 different rejectors for each metric
# keeping a named dict for each is unfeasible, so we will use a dictionary of dictionaries:
# the key will be the metric followed by the type (min, mean, max) and the value will be a dictionary with the model as key.
#  l2rejectors_distances
l2_rejectors_distances = {}
selected_l2_rejectors_distances = {}

# the values are:
# - rejection_quality_dict: the rejection quality of the model
# - plug_in_rulers: the plug-in ruler of the models
# - rejected_by_coverage: the number of samples rejected by the coverage
# - classification_quality_dict: the classification quality of the model () 
rejected_by_coverage = {}
classification_quality_dict = {}
rejection_quality_dict = {}
included_samples = {}

all_original_scores = {}
# the target coverages
target_coverages_n = 30
target_coverages = np.linspace(0.001,1,target_coverages_n-6)[::-1][1:]
target_coverages = sorted(list(set(target_coverages))+[0.5,0.9,0.95,0.99,0.01,0.05],reverse=True)
target_coverages_n = len(target_coverages)
n = len(splits['X_test'])

# initialize the dictionaries for the results
for k in models.keys():
    # Our method
    l2_rejectors[k] = cfdr.CFDistRejector(model=models[k],
                                    coverages=target_coverages)
    
    l2_rejectors_gamma[k] = cfdr.CFDistRejector(model=models[k],coverages=target_coverages)
    # The others:
    # the plug-in ruler
    plug_in_rulers[k] =  PlugInRule(model=models[k])
    # Probably I will have to change this to the version that do not refit the model
    plug_in_rulers_auc[k] = PlugInRuleAUC(model=models[k])
    all_original_scores[k] = accuracy_score(splits['y_test'], models[k].predict(splits['X_test']))
    
    # the following are the dictionaries for the results, one for each selective technique
    rejected_by_coverage[k] = {}
    classification_quality_dict[k] = {}
    rejection_quality_dict[k] = {}
    included_samples[k] = {}
    for selective_c in ["PlugInRule","PlugInRuleAUC","CFDistRejector","CFDistRejector_gamma"]:
        rejected_by_coverage[k][selective_c] = np.zeros(target_coverages_n)
        classification_quality_dict[k][selective_c] = np.zeros(target_coverages_n)
        rejection_quality_dict[k][selective_c] = np.zeros(target_coverages_n)
        included_samples[k][selective_c] = np.zeros(target_coverages_n)


for k in models.keys():
    l2_rejectors_distances[k] = {}
    for metric in focus_metrics:
        for m_type in ["min","max","mean"]:
            selective_c = "CFDistRejector_"+metric+"_"+m_type
            l2_rejectors_distances[k][selective_c] = cfdr.CFDistRejector(model=models[k],
                                    coverages=target_coverages)
                                    #distance=metric,
                                    #distance_type=m_type)
            # the following are the dictionaries for the results, one for each selective technique
            rejected_by_coverage[k][selective_c] = np.zeros(target_coverages_n)
            classification_quality_dict[k][selective_c] = np.zeros(target_coverages_n)
            rejection_quality_dict[k][selective_c] = np.zeros(target_coverages_n)
            included_samples[k][selective_c] = np.zeros(target_coverages_n)

        

metric_dicts = {
    "rejected_by_coverage":rejected_by_coverage,
    "classification_quality_dict":classification_quality_dict,
    "rejection_quality_dict":rejection_quality_dict,
    "included_samples":included_samples
}    

In [ ]:
included_samples["mlp"].keys()
selected_l2_rejectors_distances.keys()

In [ ]:
focus_metrics = ['inf',
 'chebyshev',
 'l2',
 'minkowski',
 'wasserstein',
 'l1',
 'mae',
 'sqeuclidean']

In [ ]:
for jj,k in enumerate(models.keys()):
    print("Model",k)
    ###########################
    ## SELECTIVE CLASSIFIERS ##
    ###########################
    # get the plug in rule
    plug_in_rule = plug_in_rulers[k]
    #target_coverages = [.99, .75, .50, ...] 
    plug_in_rule.calibrate(splits["X_calibration"], target_coverages=target_coverages)
    # here we create an array containing levels for acceptance depending on how the selective classifiers are calibrated
    # since we used 100 target coverages, we have 101 levels of acceptance, i.e., 0, 1, 2, 3, ..., 100
    selected_plg[k] = plug_in_rule.qband(splits["X_test"])

    
    
    plug_in_ruler_auc = plug_in_rulers_auc[k]
    # Here we calibrate the selective classifier that uses the AUC
    plug_in_ruler_auc.calibrate(X=splits["X_calibration"],
                                    y=splits["y_calibration"].values.reshape(-1),
                                    target_coverages=target_coverages)
    selected_plg_auc[k] = plug_in_rulers_auc[k].qband(splits["X_test"])

    # get the l2 rejector
    l2_rejector = l2_rejectors[k]
    # calibrate the rejector
    l2_rejector.calibrate(splits["X_calibration"], all_stats[k]["distances"]["l2"]["mean"])
    # save the selected data
    selected_l2r[k] = l2_rejector.qband(splits["X_test"],all_stats_test[k]["distances"]["l2"]["mean"])

    print(selected_plg[k][:10],type(selected_plg[k]))
    print(selected_plg_auc[k][:10],type(selected_plg_auc[k]))
    print(selected_l2r[k][:10],type(selected_l2r[k]))

    l2_rejectors_g = l2_rejectors_gamma[k]
    l2_rejectors_g.calibrate(splits["X_calibration"], all_stats[k]["distances"]["l2"]["mean"],
                             use_gamma=True)
    selected_l2r_gamma[k] = l2_rejectors_g.qband(splits["X_test"],all_stats_test[k]["distances"]["l2"]["mean"])




    for selective_k,selected in zip([
                                    "PlugInRule",
                                    "PlugInRuleAUC",
                                    "CFDistRejector",
                                    "CFDistRejector_gamma"
                                    ],
                                    [1,0,2,3]):
        if selective_k == "PlugInRuleAUC":
            selected = selected_plg_auc[k]
            selected_c = plug_in_rulers_auc[k]
        if selective_k == "PlugInRule":
            selected = selected_plg[k]
            selected_c = plug_in_rulers[k]
        if selective_k == "CFDistRejector":
            selected = selected_l2r[k]
            selected_c = l2_rejectors[k]
        if selective_k == "CFDistRejector_gamma":
            selected = selected_l2r_gamma[k]
            selected_c = l2_rejectors_g
        print(type(selected_c),type(selected))
        compute_selective_metrics(model_key=k,
                                    selected_data=selected,
                                    classifier_type=selective_k,
                                    selective_classifier=selected_c,
                                    splits=splits,
                                    target_coverages=target_coverages,
                                    n=n,
                                    metric_dicts=metric_dicts)
    # now we have to iterate over the metrics and the types of distances
    selected_l2_rejectors_distances[k] = {}
    for metric in focus_metrics:
        for m_type in ["min","max","mean"]:
            selective_c = "CFDistRejector_"+metric+"_"+m_type
            l2_rejector = l2_rejectors_distances[k][selective_c]
            l2_rejector.calibrate(splits["X_calibration"], all_stats[k]["distances"][metric][m_type])
            selected_l2_rejectors_distances[k][selective_c] = l2_rejector.qband(splits["X_test"],all_stats_test[k]["distances"][metric][m_type])
    for metric in focus_metrics:
        for m_type in ["min","max","mean"]:
            selective_c = "CFDistRejector_"+metric+"_"+m_type
            compute_selective_metrics(model_key=k,
                                    selected_data=selected_l2_rejectors_distances[k][selective_c],
                                    classifier_type=selective_c,
                                    selective_classifier=l2_rejectors_distances[k][selective_c],
                                    splits=splits,
                                    target_coverages=target_coverages,
                                    n=n,
                                    metric_dicts=metric_dicts)


In [ ]:
markers = ["o","d","s","v","^","p"]
colors = ["blue","red","green","purple"]
markers = markers*10

In [ ]:
from typing import Any, Dict
from OLD_src.Lib.L2R.code.model_agnostic import PlugInRule, PlugInRuleAUC, SCRoss
from OLD_src.MyLoreSA.metrics import nonrejected_accuracy, classification_quality, rejection_quality, rejection_classification_report


In [ ]:
#iterate over the models

fig, ax = plt.subplots(1,#len(models.keys())//2,
                       len(models.keys()),
                       figsize=(24,5))
ax = ax.flatten()
for jj,k in enumerate(models.keys()):
    for selective_k,marker,color in zip(["PlugInRule","PlugInRuleAUC","CFDistRejector","CFDistRejector_gamma"],
                                        markers,
                                        colors):
        rejected_acc = rejected_by_coverage[k][selective_k]
        ax[jj].plot(np.arange(len(rejected_acc))/len(rejected_acc), rejected_acc, 
                    label=selective_k.replace("_gamma"," $_\gamma$"),
                    marker=marker,
                    color=color)

        ax[jj].set_title(k,fontsize=15)
        ax[jj].set_xlabel('Rejection rate', fontsize=15)
        # set the xticks to be 1%, 25 % , 50 %, 75 % and 99
        ax[jj].set_xticks([0.01,0.25,0.5,0.75,0.99])
        # add the percentage sign
        ax[jj].set_xticklabels(['1%','25%','50%','75%','99%'], fontsize=13)
        ax[jj].set_ylabel('Non-rejected accuracy', fontsize=15)
        ax[jj].grid()
        # plot the 0% rejection accuracy
        low_y_lim = -0.05
        low_y_lim = min(all_original_scores.values())
        ax[jj].set_ylim(low_y_lim-0.025,1.025)
        # get the ticks for the y axis of ax, and be sure to not exceed 1.0 as tick
        ax[jj].set_yticks(np.linspace(low_y_lim,1.0,6).round(2))
        # set the font of the y ticks
        ax[jj].yaxis.set_tick_params(labelsize=13)
        # get the twin axis
        axx2 = ax[jj].twinx()
        # plot the number of included samples in the classification
        included_samp = included_samples[k][selective_k]
        axx2.plot(np.arange(target_coverages_n)/target_coverages_n, included_samp, 
                    label="% samples",
                    
                    color=color,
                    linestyle="-",alpha=0.3)
        # to be ure add the "target" rejection rate as a diagonal line

        axx2.set_ylabel("Included samples",fontsize=15,rotation=270)
        axx2.set_ylim(-0.025,1.025)
        # set the yticks to be 0, 25 % , 50 %, 75 % and 100 %
        axx2.set_yticks([0,0.25,0.5,0.75,1])
        # add the percentage sign
        axx2.set_yticklabels(['0%','25%','50%','75%','100%'], fontsize=13)
        # place grid on the y axis
        ax[jj].grid()
        #axx2.legend(loc="lower right")
        ax[jj].set_xlim(-0.05,.5)
    ax[jj].axhline(all_original_scores[k],linestyle="--",
        color="black",label=k)
    axx2.plot([0,1],[1,0],color="black",linestyle="-.",alpha=0.3)
    ax[jj].legend(loc="lower center")
    # plot the y grid
    ax[jj].grid(axis='y')
plt.suptitle("Non-rejected accuracy",fontsize=20)
plt.tight_layout()
for k in models.keys():
    # roc = roc_auc_score(splits['y_test'], models[k].predict_proba(splits['X_test'])[:,1]).round(2)
    # print(f"Model {k} ROC AUC: {roc}")
    acc = accuracy_score(splits['y_test'], models[k].predict(splits['X_test'])).round(4)
    print(f"Model {k} Accuracy: {acc}")
plt.show()

# TODO add other metrics